# Torob RAG — Phase 1 Evaluation Dashboard

این نوت‌بوک برای ارائه‌ی فاز ۱ ساخته شده و **metrics** و **artifact**های اصلی را از مسیر `torob_rag/artifacts/` می‌خواند.

✅ **نحوه اجرا**: از ریشه پروژه (جایی که پوشه `torob_rag/` قرار دارد) نوت‌بوک را اجرا کنید.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('../..').resolve()
ART = PROJECT_ROOT / 'torob_rag' / 'artifacts'

def load_json(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    return json.loads(path.read_text(encoding='utf-8'))

def safe_read_csv(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    return pd.read_csv(path)

print('Project root:', PROJECT_ROOT)
print('Artifacts dir:', ART)


## 1) Text-only Retrieval — Evaluation

این بخش `eval_summary.json` و نمونه‌ی خطاها را نمایش می‌دهد.


In [ ]:
text_eval_path = ART / 'eval' / 'text' / 'eval_summary.json'
text_eval = load_json(text_eval_path)
text_eval

In [ ]:
metrics_mean = pd.Series(text_eval['metrics_mean']).sort_index()
metrics_std = pd.Series(text_eval.get('metrics_std', {})).sort_index()

display(pd.DataFrame({'mean': metrics_mean, 'std': metrics_std}))


In [ ]:
# Plot Recall@k, MRR@k, NDCG@k
def plot_metric(prefix: str, title: str):
    items = [(k, v) for k, v in text_eval['metrics_mean'].items() if k.startswith(prefix)]
    items = sorted(items, key=lambda x: int(x[0].split('@')[1]))
    ks = [int(k.split('@')[1]) for k, _ in items]
    vals = [v for _, v in items]
    plt.figure()
    plt.plot(ks, vals, marker='o')
    plt.xticks(ks)
    plt.ylim(0, max(vals) * 1.3 if max(vals) > 0 else 1)
    plt.title(title)
    plt.xlabel('k')
    plt.ylabel(prefix.replace('@', '') + ' value')
    plt.grid(True)
    plt.show()

plot_metric('recall@', 'Text-only: Recall@k')
plot_metric('mrr@', 'Text-only: MRR@k')
plot_metric('ndcg@', 'Text-only: NDCG@k')


### 1.1) Failure Examples (Readable)

این جدول برای ارائه خیلی کاربردی است: نشان می‌دهد query چه بوده و مدل چه چیزی را top-1 برگردانده.


In [ ]:
fail_path = ART / 'eval' / 'text' / 'failure_examples_readable.csv'
fail = safe_read_csv(fail_path)
fail.head(20)

In [ ]:
# چند آمار سریع روی خطاها
fail['query_len'] = fail['query'].fillna('').astype(str).str.len()
summary = fail[['query_len', 'num_relevant_in_index']].describe().T
display(summary)

plt.figure()
plt.hist(fail['query_len'], bins=30)
plt.title('Failure examples: query length distribution')
plt.xlabel('query length')
plt.ylabel('count')
plt.grid(True)
plt.show()


## 2) Image-only Retrieval — CLIP + FAISS

در نبود لاگ‌های image-query، یک **sanity evaluation** داریم (self-retrieval). این یعنی pipeline و index صحیح هستند.


In [ ]:
img_sanity_path = ART / 'eval' / 'image' / 'clip_v1' / 'sanity_report.json'
img_sanity = load_json(img_sanity_path)
img_sanity

## 3) Multimodal Retrieval — Late Fusion

این مدل با late-fusion پیاده‌سازی شد:
- بازیابی جداگانه‌ی text و image
- ترکیب امتیازها با وزن `alpha_text`

### نکته مهم
`mm_sanity_eval` یک sanity است (query=embedding خود آیتم). هدفش اثبات correctness pipeline است، نه سنجش تجربه‌ی کاربر.


In [ ]:
mm_sanity_path = ART / 'eval' / 'multimodal' / 'clip_v1' / 'mm_sanity_eval.json'
mm_sanity = load_json(mm_sanity_path)
mm_sanity

## 4) Download / Coverage Summary (Image)

برای ارائه مهم است که بگوییم از 20k تصویر، چندتا واقعاً دانلود و embed شدند.


In [ ]:
dl_summary_path = ART / 'images' / 'clip_v1' / 'download_summary.json'
if dl_summary_path.exists():
    dl = load_json(dl_summary_path)
    display(dl)
else:
    print('download_summary.json not found (optional).')

img_meta_path = ART / 'embeddings' / 'clip_v1' / 'image' / 'image_embeddings_meta.json'
if img_meta_path.exists():
    img_meta = load_json(img_meta_path)
    display(img_meta)
else:
    print('image_embeddings_meta.json not found.')


## 5) Executive Summary (برای اسلاید آخر)

این جدول را می‌توانید مستقیم در ارائه نشان دهید.


In [ ]:
rows = []

# Text metrics
rows.append({
    'system': 'Text-only',
    'index_size': text_eval.get('index_size'),
    'recall@10': text_eval['metrics_mean'].get('recall@10'),
    'mrr@10': text_eval['metrics_mean'].get('mrr@10'),
    'ndcg@10': text_eval['metrics_mean'].get('ndcg@10'),
    'notes': 'Eval on 5k samples from click logs (in-index).'
})

# Image sanity
rows.append({
    'system': 'Image-only (CLIP)',
    'index_size': img_sanity.get('n_items'),
    'recall@10': None,
    'mrr@10': None,
    'ndcg@10': None,
    'notes': f"Sanity self_recall@1={img_sanity.get('self_recall@1'):.4f}"
})

# Multimodal sanity
mm = mm_sanity
rows.append({
    'system': 'Multimodal (Late Fusion)',
    'index_size': None,
    'recall@10': mm.get('metrics_mean', {}).get('recall@10'),
    'mrr@10': mm.get('metrics_mean', {}).get('mrr@10'),
    'ndcg@10': None,
    'notes': f"Sanity (self-embedding), alpha_text={mm.get('alpha_text')}"
})

summary_df = pd.DataFrame(rows)
summary_df